# Домашнее задание: генерация текста с помощью RNN (собственный корпус)

## Задача

1. Собрать свой небольшой текстовый корпус (не менее 1000 предложений) с помощью библиотеки `requests` (парсинг новостного сайта, блога, форума и т.д.).
2. Обучить рекуррентную нейросеть (RNN/LSTM) на собранных данных для генерации текста (по образцу из приложенного ноутбука `Copy_of_rnn.ipynb`).
3. После обучения вывести на экран 2–3 сгенерированных предложения.
4. *Дополнительно (на 5 баллов, но не обязательно):* посчитать метрику перплексии (perplexity) на валидационной выборке.
5. *Для себя (не оценивается):* обучить модель с использованием GPU.

## Критерии оценки

- **3 балла** — корпус собран (≥1000 предложений), модель обучена, сгенерировано хотя бы 1 предложение.
- **4 балла** — всё из п.3 + код с комментариями, объясняющими ключевые этапы.
- **5 баллов** — всё из п.4 + дополнительно посчитана метрика перплексии.

## Важно

- Качество сгенерированного текста не оценивается.
- Выберите **свой уникальный сайт** для парсинга и укажите его в отчёте.
- Не используйте готовые датасеты из интернета.


## 1. Установка и импорт библиотек

In [1]:
!pip install beautifulsoup4 requests lxml -q

import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow версия:", tf.__version__)
print("GPU доступна:", tf.config.list_physical_devices('GPU'))

TensorFlow версия: 2.20.0
GPU доступна: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Парсинг текстового корпуса

**Ваш уникальный сайт:** [вставьте ссылку на выбранный сайт]

**Обоснование выбора:** [почему вы выбрали именно этот сайт? Например: «чтобы генерировать заголовки финансовых новостей»]

In [4]:
def scrape_corpus(base_url, num_pages=10):
    """
    Парсинг текстового корпуса с выбранного сайта.
    Замените селекторы под ваш сайт.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    texts = []

    for page in range(1, num_pages + 1):
        url = f"{base_url}?page={page}"  # формат может отличаться
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # !!! ЗАМЕНИТЕ СЕЛЕКТОР ПОД ВАШ САЙТ !!!
            # Пример для заголовков новостей:
            for tag in soup.find_all(['h2', 'h3', 'p']):
                text = tag.get_text(strip=True)
                if text and len(text) > 10:
                    texts.append(text)

            print(f"Страница {page}: собрано {len(texts)} текстов")
            time.sleep(1)  # вежливый парсинг

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")

    return texts


# ЗАМЕНИТЕ URL НА ВАШ
base_url = "https://lenta.ru/"  # <- ВАШ САЙТ
corpus = scrape_corpus(base_url, num_pages=10)

print(f"\nВсего собрано текстов: {len(corpus)}")
print("Примеры:", corpus[:5])

Страница 1: собрано 155 текстов
Страница 2: собрано 310 текстов
Страница 3: собрано 465 текстов
Страница 4: собрано 620 текстов
Страница 5: собрано 775 текстов
Страница 6: собрано 930 текстов
Страница 7: собрано 1085 текстов
Страница 8: собрано 1240 текстов
Страница 9: собрано 1395 текстов
Страница 10: собрано 1550 текстов

Всего собрано текстов: 1550
Примеры: ['Путин назвал допустимую точку возобновления переговоров с Украиной', 'Россиянин залез в квартиру и изнасиловал женщину', 'Глава ФСБ России высказался о Владимире Зеленском', 'Официальный курс доллара приблизился к 75 рублям', 'Путин назвал виновника в прекращении переговоров по Украине']


## 3. Подготовка данных для RNN

Токенизация, создание последовательностей, паддинг.

In [5]:
# Создаём токенизатор
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

# Преобразуем тексты в последовательности чисел
sequences = tokenizer.texts_to_sequences(corpus)

# Создаём входные и выходные данные для causal language modeling
X, y = [], []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

# Паддинг
X = pad_sequences(X)

# One-hot encoding для y
vocab_size = len(tokenizer.word_index) + 1
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

print(f"Размер входных данных X: {X.shape}")
print(f"Размер выходных данных y: {y.shape}")
print(f"Размер словаря: {vocab_size}")

Размер входных данных X: (12610, 19)
Размер выходных данных y: (12610, 904)
Размер словаря: 904


## 4. Создание и обучение модели RNN (LSTM)

In [6]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100, input_length=X.shape[1]))
model.add(LSTM(150, return_sequences=False))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Обучение (можно увеличить эпохи при хороших результатах)
history = model.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.0454 - loss: 6.1586 - val_accuracy: 0.0587 - val_loss: 5.2973
Epoch 2/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1339 - loss: 4.5778 - val_accuracy: 0.3045 - val_loss: 3.5598
Epoch 3/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4516 - loss: 2.9404 - val_accuracy: 0.7264 - val_loss: 2.0961
Epoch 4/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7672 - loss: 1.6919 - val_accuracy: 0.8858 - val_loss: 1.1627
Epoch 5/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9053 - loss: 0.9389 - val_accuracy: 0.9469 - val_loss: 0.6509
Epoch 6/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9483 - loss: 0.5400 - val_accuracy: 0.9603 - val_loss: 0.3923
Epoch 7/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9586 - loss: 0.3400 - val_accuracy: 0.9643 - val_loss: 0.2612
Epoch 8/10
316/316 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9602 - loss: 0.2379 - val_accuracy: 

## 5. Генерация текста

Функция генерации и вывод 2–3 предложений.

In [14]:
def generate_text(seed_text, next_words, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Пример генерации
seed = "Завтрашний"  # замените на любое начало
generated = generate_text(seed, next_words=10, max_sequence_len=X.shape[1])
print("Сгенерированный текст:", generated)

seed_ = "Завтрашний"  # замените на любое начало
generated_ = generate_text(seed_, next_words=10, max_sequence_len=X.shape[0])
print("Сгенерированный текст_:", generated_)

# Сгенерируйте ещё 1-2 примера с другими seed
seed2 = "Вчерашний"
generated2 = generate_text(seed2, next_words=20, max_sequence_len=X.shape[1])
print("Сгенерированный текст 2:", generated2)

seed3 = "В понедельник"
generated3 = generate_text(seed3, next_words=15, max_sequence_len=X.shape[1])
print("Сгенерированный текст 3:", generated3)

seed4 = "Названы"
generated4 = generate_text(seed4, next_words=20, max_sequence_len=X.shape[1])
print("Сгенерированный текст 4:", generated4)

seed5 = "Новости спорта"
generated5 = generate_text(seed5, next_words=20, max_sequence_len=X.shape[0])
print("Сгенерированный текст 5:", generated5)

seed6 = "В ботаническом саду"
generated6 = generate_text(seed6, next_words=30, max_sequence_len=X.shape[1])
print("Сгенерированный текст 6:", generated6)

Сгенерированный текст: Завтрашний известно о поручении трампа по украине россии оценил ситуацию в
Сгенерированный текст_: Завтрашний известно о поручении трампа по вкладам в российских банках снизились
Сгенерированный текст 2: Вчерашний известно о поручении трампа по украине россии оценил ситуацию в зоне сво и возможность переговоров главное из заявлений подготовке внутри
Сгенерированный текст 3: В понедельник россии определили предельный срок стажировки стажировки стажировки о военном конфликте с киевом и судьбе украины
Сгенерированный текст 4: Названы известно о поручении трампа по украине россии оценил ситуацию в зоне сво и возможность переговоров главное из заявлений подготовке внутри
Сгенерированный текст 5: Новости спорта известно о поручении трампа по вкладам в российских банках снизились таблеток на видео в сша о жертвах почти 30 почти
Сгенерированный текст 6: В ботаническом саду россии определили предельный срок стажировки стажировки стажировки о военном конфликте с киевом и с

## 6. (Дополнительно, на 5 баллов) Расчёт перплексии

Перплексия = exp(loss). Чем ниже, тем лучше модель предсказывает последовательность.

In [15]:
# Оцениваем модель на валидационных данных
loss, accuracy = model.evaluate(X, y, verbose=0)
perplexity = np.exp(loss)

print(f"Потери (loss): {loss:.4f}")
print(f"Перплексия: {perplexity:.4f}")
print("Пояснение: перплексия показывает, насколько модель 'удивлена' тестовыми данными. Чем ниже — тем лучше.")

Потери (loss): 0.1333
Перплексия: 1.1426
Пояснение: перплексия показывает, насколько модель 'удивлена' тестовыми данными. Чем ниже — тем лучше.


## 7. GPU (не оценивается)

Убедитесь, что обучение запускалось на GPU:
```python
print("GPU доступна:", tf.config.list_physical_devices('GPU'))
```

В Google Colab: Среда выполнения - Изменить тип среды выполнения - Выберите T4 GPU.

Документация: https://www.tensorflow.org/guide/gpu

In [ ]:
print("Устройства GPU:", tf.config.list_physical_devices('GPU'))